# Laboratório de Ciência de Dados
## Aula 6 — Agrupamento (Clustering): implementação do K-Means e análise dos grupos obtidos

### Situação

Nas aulas anteriores, trabalhamos principalmente com **aprendizado supervisionado**: existia uma variável-alvo conhecida e podíamos medir diretamente se uma previsão estava correta ou incorreta.

Agora a situação muda.

A **VGAbank** possui dados sobre o comportamento de seus clientes, mas não possui uma coluna dizendo a qual “tipo de cliente” cada pessoa pertence.

A pergunta é:

> **É possível descobrir grupos de clientes com comportamentos semelhantes sem possuir rótulos previamente definidos?**

Esse é um problema de **aprendizado não supervisionado**.

### O que você vai fazer

Nesta aula, você irá:

- compreender o funcionamento do **K-Means**;
- implementar uma versão didática do algoritmo;
- observar por que a **escala dos atributos** influencia o agrupamento;
- aplicar `KMeans` do `scikit-learn`;
- utilizar **inércia** e **silhouette** para investigar valores de \(k\);
- visualizar os grupos com auxílio de **PCA**;
- construir um **perfil dos clusters**;
- interpretar os grupos sem tratá-los como classes reais;
- reconhecer limitações do K-Means.

> O objetivo não é apenas obter números de cluster. O resultado só se torna útil quando conseguimos **explicar o que caracteriza cada grupo**.

# 1. Preparando o ambiente

Utilizaremos `pandas`, `numpy`, `matplotlib` e ferramentas do `scikit-learn`.

As células de código estão comentadas para que você acompanhe não apenas o resultado, mas também a lógica da implementação.

In [ ]:
# Operações numéricas.
import numpy as np

# Manipulação de tabelas.
import pandas as pd

# Visualizações.
import matplotlib.pyplot as plt

# Padronização das variáveis.
from sklearn.preprocessing import StandardScaler

# Implementação pronta do K-Means.
from sklearn.cluster import KMeans

# Métricas para avaliação interna de clusters.
from sklearn.metrics import silhouette_score, silhouette_samples

# Redução de dimensionalidade para visualização.
from sklearn.decomposition import PCA

# Mantém os resultados reproduzíveis.
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 2. O que muda em relação à classificação?

Em classificação, normalmente temos algo como:

```text
atributos → modelo → classe conhecida
```

Por exemplo:

```text
transação → modelo → fraude / legítima
```

No agrupamento não existe uma classe conhecida durante o treinamento.

Temos:

```text
atributos → algoritmo → grupos encontrados nos dados
```

O algoritmo procura observações **semelhantes entre si** segundo algum critério matemático.

No K-Means, essa semelhança está diretamente relacionada à **distância euclidiana**.

## 2.1 A ideia do K-Means

O K-Means procura dividir as observações em `k` grupos.

O procedimento básico é:

1. escolher `k` centroides iniciais;
2. atribuir cada observação ao centroide mais próximo;
3. recalcular cada centroide como a média dos pontos de seu grupo;
4. repetir os passos 2 e 3 até que as atribuições parem de mudar ou a alteração se torne muito pequena.

O algoritmo tenta reduzir a soma das distâncias quadráticas das observações aos centroides de seus respectivos grupos. Essa quantidade está relacionada à **inércia**, que veremos mais adiante.

# 3. Implementando uma versão didática do K-Means

Antes de utilizar a implementação pronta do `scikit-learn`, vamos construir uma versão simplificada.

Usaremos uma base bidimensional pequena apenas para visualizar o raciocínio do algoritmo.

In [ ]:
# Criamos três regiões de pontos apenas para fins didáticos.
grupo_a = rng.normal(loc=[2, 2], scale=[0.45, 0.45], size=(20, 2))
grupo_b = rng.normal(loc=[6, 6], scale=[0.50, 0.50], size=(20, 2))
grupo_c = rng.normal(loc=[2, 7], scale=[0.55, 0.45], size=(20, 2))

# Empilha os três conjuntos em uma única matriz.
X_didatico = np.vstack([grupo_a, grupo_b, grupo_c])

# Visualiza os dados sem informar ao algoritmo quantos grupos foram usados na geração.
plt.figure(figsize=(6, 5))
plt.scatter(X_didatico[:, 0], X_didatico[:, 1])
plt.xlabel("Atributo 1")
plt.ylabel("Atributo 2")
plt.title("Dados antes do agrupamento")
plt.grid(alpha=0.2)
plt.show()

Visualmente, parecem existir regiões distintas.

Mas o algoritmo recebe apenas os pontos. Ele não recebe os nomes `grupo_a`, `grupo_b` ou `grupo_c`.

Vamos implementar as principais etapas do K-Means.

In [ ]:
def kmeans_didatico(X, k, max_iter=100, random_state=42):
    # Cria um gerador de números aleatórios reprodutível.
    rng_local = np.random.default_rng(random_state)

    # Escolhe k observações diferentes como centroides iniciais.
    indices_iniciais = rng_local.choice(len(X), size=k, replace=False)
    centroides = X[indices_iniciais].copy()

    # Repetimos o processo até convergir ou atingir max_iter.
    for iteracao in range(max_iter):
        # ----------------------------------------------------
        # ETAPA 1 — atribuição
        # ----------------------------------------------------
        # Para cada observação, calculamos sua distância
        # a cada um dos centroides.
        distancias = np.linalg.norm(
            X[:, np.newaxis, :] - centroides[np.newaxis, :, :],
            axis=2
        )

        # O cluster de cada observação é o centroide mais próximo.
        rotulos = np.argmin(distancias, axis=1)

        # ----------------------------------------------------
        # ETAPA 2 — atualização
        # ----------------------------------------------------
        novos_centroides = []

        for cluster in range(k):
            # Seleciona todas as observações atribuídas ao cluster.
            pontos_cluster = X[rotulos == cluster]

            # Se o cluster não estiver vazio,
            # seu novo centroide é a média de seus pontos.
            if len(pontos_cluster) > 0:
                novo = pontos_cluster.mean(axis=0)
            else:
                # Se estiver vazio, mantemos o centroide anterior.
                novo = centroides[cluster]

            novos_centroides.append(novo)

        novos_centroides = np.array(novos_centroides)

        # ----------------------------------------------------
        # ETAPA 3 — convergência
        # ----------------------------------------------------
        # np.allclose verifica se os centroides praticamente
        # deixaram de mudar.
        if np.allclose(centroides, novos_centroides):
            centroides = novos_centroides
            break

        centroides = novos_centroides

    # Recalcula os rótulos usando os centroides finais.
    distancias_finais = np.linalg.norm(
        X[:, np.newaxis, :] - centroides[np.newaxis, :, :],
        axis=2
    )
    rotulos_finais = np.argmin(distancias_finais, axis=1)

    return rotulos_finais, centroides, iteracao + 1

In [ ]:
# Aplicamos nossa implementação com k = 3.
rotulos_didaticos, centroides_didaticos, n_iter = kmeans_didatico(
    X_didatico,
    k=3,
    random_state=RANDOM_STATE
)

print("Número de iterações até a convergência:", n_iter)
print("\nCentroides encontrados:")
print(np.round(centroides_didaticos, 2))

# Visualização do resultado.
plt.figure(figsize=(6, 5))
plt.scatter(
    X_didatico[:, 0],
    X_didatico[:, 1],
    c=rotulos_didaticos
)

# Os X representam os centroides finais.
plt.scatter(
    centroides_didaticos[:, 0],
    centroides_didaticos[:, 1],
    marker="X",
    s=180
)

plt.xlabel("Atributo 1")
plt.ylabel("Atributo 2")
plt.title("Grupos encontrados pela implementação didática")
plt.grid(alpha=0.2)
plt.show()

### Pergunta 1

Os grupos encontrados correspondem aproximadamente às regiões visuais que você identificou antes?

O algoritmo “sabia” previamente o significado dos grupos?

> Os números `0`, `1` e `2` são apenas identificadores. **Cluster 0 não significa automaticamente “grupo pior”, “grupo menor” ou “primeiro grupo real”.**

# 4. Problema aplicado: segmentação de clientes

A VGAbank deseja compreender diferentes padrões de relacionamento com o banco.

Vamos trabalhar com:

- `renda_mensal`;
- `gasto_cartao`;
- `transacoes_mes`;
- `recencia_dias`: dias desde a última transação;
- `saldo_medio`;
- `uso_app_mes`: quantidade aproximada de acessos mensais ao aplicativo.

A base será simulada, mas os atributos foram construídos para permitir uma análise semelhante à de um problema real de segmentação.

In [ ]:
def gerar_clientes(n, medias, desvios):
    # Gera as colunas usando médias e desvios definidos.
    return rng.normal(
        loc=np.array(medias),
        scale=np.array(desvios),
        size=(n, len(medias))
    )


# Criamos quatro perfis comportamentais diferentes.
segmento_1 = gerar_clientes(
    230,
    medias=[4500, 1200, 9, 38, 3500, 6],
    desvios=[900, 450, 3, 12, 1400, 3]
)

segmento_2 = gerar_clientes(
    240,
    medias=[8500, 3600, 23, 12, 13000, 18],
    desvios=[1400, 900, 5, 5, 3500, 5]
)

segmento_3 = gerar_clientes(
    210,
    medias=[16000, 7200, 28, 6, 42000, 24],
    desvios=[2600, 1500, 6, 3, 8000, 6]
)

segmento_4 = gerar_clientes(
    220,
    medias=[6000, 3000, 38, 4, 6500, 48],
    desvios=[1100, 750, 7, 2, 1800, 8]
)

# Junta os quatro blocos.
dados_clientes = np.vstack([
    segmento_1,
    segmento_2,
    segmento_3,
    segmento_4
])

colunas = [
    "renda_mensal",
    "gasto_cartao",
    "transacoes_mes",
    "recencia_dias",
    "saldo_medio",
    "uso_app_mes"
]

df = pd.DataFrame(dados_clientes, columns=colunas)

# Valores negativos não fazem sentido neste contexto.
for coluna in colunas:
    df[coluna] = df[coluna].clip(lower=0)

# Arredondamentos apenas para facilitar a leitura.
df["renda_mensal"] = df["renda_mensal"].round(2)
df["gasto_cartao"] = df["gasto_cartao"].round(2)
df["transacoes_mes"] = df["transacoes_mes"].round(0)
df["recencia_dias"] = df["recencia_dias"].round(0)
df["saldo_medio"] = df["saldo_medio"].round(2)
df["uso_app_mes"] = df["uso_app_mes"].round(0)

df.head()

In [ ]:
# Inspeciona dimensões e estatísticas básicas.
print("Dimensões da base:", df.shape)

df.describe().round(2)

### Pergunta 2

Compare as ordens de grandeza.

`saldo_medio` pode assumir valores de dezenas de milhares, enquanto `recencia_dias` e `uso_app_mes` estão em dezenas.

Se utilizarmos diretamente a distância euclidiana, quais atributos tenderão a exercer maior influência?

Essa diferença de escala significa que esses atributos são necessariamente mais importantes?

# 5. Por que a padronização é importante?

O K-Means utiliza distâncias.

Se um atributo possui valores numericamente muito maiores que outro, ele pode dominar a distância mesmo sem ser conceitualmente mais importante.

Por isso, em muitos problemas de clustering, é adequado padronizar os atributos.

Utilizaremos o **Z-score**:

$$
z = \frac{x-\mu}{\sigma}
$$

Após a transformação, os atributos ficam aproximadamente com média 0 e desvio padrão 1.

In [ ]:
# Cria o objeto responsável pela padronização.
scaler = StandardScaler()

# fit_transform():
# 1. calcula média e desvio padrão de cada coluna;
# 2. transforma os valores.
X_padronizado = scaler.fit_transform(df[colunas])

# Converte novamente para DataFrame apenas para inspeção.
df_padronizado = pd.DataFrame(
    X_padronizado,
    columns=colunas,
    index=df.index
)

df_padronizado.describe().round(2)

## 5.1 Comparando K-Means com e sem padronização

Vamos executar o algoritmo nos dois casos usando o mesmo `k`.

A intenção não é declarar um agrupamento “correto” apenas olhando números, mas observar que **a representação dos dados altera o resultado**.

In [ ]:
# K-Means nos dados ORIGINAIS.
kmeans_sem_escala = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=20
)
rotulos_sem_escala = kmeans_sem_escala.fit_predict(df[colunas])

# K-Means nos dados PADRONIZADOS.
kmeans_padronizado = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=20
)
rotulos_padronizados = kmeans_padronizado.fit_predict(X_padronizado)

print("Quantidade de clientes por cluster — sem padronização:")
print(pd.Series(rotulos_sem_escala).value_counts().sort_index())

print("\nQuantidade de clientes por cluster — com padronização:")
print(pd.Series(rotulos_padronizados).value_counts().sort_index())

### Pergunta 3

Os agrupamentos foram idênticos?

Mesmo que ambos tenham quatro clusters, isso não significa que as mesmas pessoas foram colocadas juntas.

A padronização não é uma etapa “obrigatória” para qualquer problema, mas deve ser considerada sempre que o método depende de distância e os atributos possuem escalas diferentes.

# 6. Quantos clusters devemos utilizar?

Até agora definimos `k=4` porque queríamos demonstrar o funcionamento.

Em um problema real, frequentemente não sabemos quantos grupos existem.

Duas ferramentas comuns são:

- **inércia**;
- **silhouette**.

Nenhuma delas deve ser tratada como uma regra automática.

## 6.1 Inércia e método do cotovelo

A inércia é a soma das distâncias quadráticas de cada observação ao centroide do seu cluster.

Quanto menor, mais próximos os pontos estão dos respectivos centroides.

Porém, aumentar `k` quase sempre reduz a inércia. Por isso procuramos uma região em que adicionar novos clusters passa a produzir ganhos menores: o chamado **cotovelo**.

In [ ]:
resultados_k = []

# Testaremos valores de k entre 2 e 8.
for k in range(2, 9):
    modelo = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20
    )

    # Ajusta o modelo e obtém os rótulos.
    rotulos = modelo.fit_predict(X_padronizado)

    # inertia_ é calculada pelo próprio KMeans.
    inercia = modelo.inertia_

    # A silhouette mede coesão e separação dos grupos.
    silhueta = silhouette_score(X_padronizado, rotulos)

    resultados_k.append({
        "k": k,
        "inercia": inercia,
        "silhouette": silhueta
    })

df_k = pd.DataFrame(resultados_k)
df_k.round(4)

In [ ]:
# Gráfico do método do cotovelo.
plt.figure(figsize=(7, 4))
plt.plot(df_k["k"], df_k["inercia"], marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inércia")
plt.title("Método do cotovelo")
plt.xticks(df_k["k"])
plt.grid(alpha=0.3)
plt.show()

### Pergunta 4

Em qual região do gráfico você observa uma mudança mais clara na taxa de redução da inércia?

Existe necessariamente um único “cotovelo” indiscutível?

# 7. Silhouette

A **silhouette** considera duas ideias:

- quão próximo um ponto está das observações do próprio cluster;
- quão distante está das observações do cluster vizinho mais próximo.

De forma geral:

- próximo de **1** → observação bem inserida em seu grupo;
- próximo de **0** → observação está na fronteira entre grupos;
- valor **negativo** → pode estar mais próxima de outro grupo do que do próprio.

O `silhouette_score` calcula a média para todas as observações.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(df_k["k"], df_k["silhouette"], marker="o")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette média")
plt.title("Silhouette para diferentes valores de k")
plt.xticks(df_k["k"])
plt.grid(alpha=0.3)
plt.show()

### Pergunta 5

Qual valor de `k` apresentou a maior silhouette média?

Esse valor coincide com a região sugerida pelo método do cotovelo?

> A escolha de `k` não deve depender apenas de maximizar uma métrica. Também precisamos verificar se os grupos são **interpretáveis e úteis no contexto do problema**.

# 8. Ajustando o K-Means escolhido

Para esta base, adotaremos `k=4`.

A escolha é coerente com a estrutura observada nas métricas e permitirá trabalhar a interpretação dos segmentos.

In [ ]:
# Cria o modelo final.
kmeans = KMeans(
    n_clusters=4,
    random_state=RANDOM_STATE,
    n_init=20
)

# Ajusta o modelo e obtém o cluster de cada cliente.
clusters = kmeans.fit_predict(X_padronizado)

# Cria uma cópia da base original para preservar os dados.
df_resultado = df.copy()

# Adiciona o identificador do cluster.
df_resultado["cluster"] = clusters

df_resultado.head()

In [ ]:
# Quantidade de observações em cada grupo.
tamanho_clusters = (
    df_resultado["cluster"]
    .value_counts()
    .sort_index()
    .rename("clientes")
    .to_frame()
)

tamanho_clusters

Os números dos clusters são arbitrários.

Se executarmos outro algoritmo ou mudarmos a inicialização, um grupo chamado `cluster 0` poderia passar a se chamar `cluster 2`.

Por isso, o número do cluster não deve ser interpretado como uma categoria com ordem natural.

# 9. O passo mais importante: interpretar os clusters

Até agora o algoritmo retornou apenas números.

Para transformar o agrupamento em informação útil, precisamos estudar o **perfil de cada grupo**.

Uma primeira abordagem é calcular a média dos atributos originais dentro de cada cluster.

In [ ]:
# groupby("cluster") separa os clientes por grupo.
# mean() calcula a média de cada atributo dentro do cluster.
perfil_clusters = (
    df_resultado
    .groupby("cluster")[colunas]
    .mean()
    .round(1)
)

perfil_clusters

### Pergunta 6

Observe a tabela e tente descrever cada cluster **sem utilizar ainda nomes comerciais**.

Para cada grupo, responda:

- renda é relativamente baixa, média ou alta?
- gasto no cartão é baixo ou alto?
- clientes movimentam a conta com frequência?
- a recência é pequena ou grande?
- saldo médio é baixo ou alto?
- uso do aplicativo é intenso ou reduzido?

Essa descrição deve partir dos dados, e não de estereótipos.

## 9.1 Perfil relativo à média da população

Às vezes é mais fácil perceber as diferenças quando comparamos cada cluster com a média geral.

Vamos calcular:

$$
\text{índice relativo}=\frac{\text{média do cluster}}{\text{média geral}}
$$

Valores:

- maiores que 1 → acima da média geral;
- menores que 1 → abaixo da média geral.

In [ ]:
# Média geral de cada atributo.
media_geral = df_resultado[colunas].mean()

# Divide a média de cada cluster pela média geral.
perfil_relativo = (
    perfil_clusters / media_geral
).round(2)

perfil_relativo

In [ ]:
# Transpomos a tabela para facilitar o gráfico:
# atributos nas linhas e clusters nas séries.
perfil_relativo.T.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.axhline(1, linestyle="--")
plt.ylabel("Razão em relação à média geral")
plt.xlabel("Atributo")
plt.title("Perfil relativo dos clusters")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 10. Visualização com PCA

Temos seis atributos. Não é possível representá-los diretamente em um gráfico bidimensional sem perder informação.

Usaremos **PCA (Análise de Componentes Principais)** apenas para criar uma projeção em duas dimensões.

Importante:

> o K-Means já foi calculado usando os seis atributos padronizados. O PCA será utilizado aqui principalmente para **visualização**, não para redefinir os clusters.

In [ ]:
# Cria uma PCA com duas componentes.
pca = PCA(n_components=2)

# Projeta os seis atributos padronizados em duas dimensões.
X_pca = pca.fit_transform(X_padronizado)

# Percentual da variância representado por cada componente.
variancia = pca.explained_variance_ratio_

print(
    "Variância explicada pelas duas componentes:",
    round(variancia.sum() * 100, 2),
    "%"
)

In [ ]:
# Projeta também os centroides para o espaço bidimensional da PCA.
centroides_pca = pca.transform(kmeans.cluster_centers_)

plt.figure(figsize=(7, 5))

# Pontos identificados pelo cluster atribuído.
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=clusters,
    alpha=0.6
)

# Marca os centroides projetados.
plt.scatter(
    centroides_pca[:, 0],
    centroides_pca[:, 1],
    marker="X",
    s=220
)

plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Visualização dos clusters por PCA")
plt.grid(alpha=0.2)
plt.show()

### Pergunta 7

Os clusters parecem visualmente separados na projeção?

Se dois grupos se sobrepõem no gráfico, isso prova que são idênticos?

Não necessariamente: o gráfico mostra apenas uma projeção bidimensional de um espaço originalmente com seis dimensões.

# 11. Avaliando a coesão dos grupos

A silhouette média resume todo o agrupamento, mas também podemos observar a média da silhouette **por cluster**.

Isso ajuda a identificar grupos que podem estar menos bem definidos.

In [ ]:
# Calcula a silhouette individual de cada cliente.
silhuetas_individuais = silhouette_samples(
    X_padronizado,
    clusters
)

# Salva os valores no DataFrame apenas para análise.
df_resultado["silhouette"] = silhuetas_individuais

# Média, mínimo, máximo e tamanho em cada cluster.
silhouette_por_cluster = (
    df_resultado
    .groupby("cluster")["silhouette"]
    .agg(["mean", "min", "max", "count"])
    .round(3)
)

silhouette_por_cluster

In [ ]:
# Visualiza a silhouette média de cada cluster.
silhouette_por_cluster["mean"].plot(
    kind="bar",
    figsize=(7, 4)
)

plt.xlabel("Cluster")
plt.ylabel("Silhouette média")
plt.title("Coesão/separação média por cluster")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Pergunta 8

Existe algum cluster com silhouette média claramente inferior aos demais?

O que isso poderia indicar sobre sua separação em relação aos grupos vizinhos?

# 12. Do número para uma interpretação

Os identificadores `0`, `1`, `2` e `3` não possuem significado por si mesmos.

Depois de analisar os perfis, um analista pode criar **nomes descritivos** para facilitar a comunicação.

Por exemplo:

- “baixo relacionamento”;
- “relacionamento intermediário”;
- “alto valor”;
- “digital intensivo”.

Mas o nome deve ser consequência das características observadas. Não devemos primeiro inventar uma categoria e depois forçar os dados a caber nela.

### Pergunta 9 — nomeando os segmentos

Crie um nome descritivo para cada cluster e justifique com **pelo menos três atributos**.

| Cluster | Nome proposto | Evidências |
|---|---|---|
| 0 |  |  |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

Evite nomes depreciativos ou que atribuam características pessoais que não estão nos dados.

# 13. Limitações do K-Means

**1. Precisamos definir `k`.**  
O número de clusters não é descoberto automaticamente.

**2. O resultado depende da escala.**  
A distância euclidiana é sensível às unidades dos atributos.

**3. O algoritmo tende a funcionar melhor com grupos aproximadamente compactos e convexos.**  
Estruturas muito irregulares podem ser mal representadas.

**4. Outliers podem deslocar os centroides.**  
Como os centroides são médias, observações extremas podem ter impacto importante.

**5. Os grupos não são “verdades naturais”.**  
Clusters são uma representação produzida por um algoritmo, atributos e decisões de modelagem.

**6. Resultados precisam de validação substantiva.**  
Uma boa silhouette não garante que os grupos sejam úteis para o negócio ou cientificamente relevantes.

# 14. O que acontece quando escolhemos outro valor de `k`?

Vamos comparar rapidamente `k=2`, `k=4` e `k=7`.

Observe que o algoritmo sempre produzirá a quantidade de clusters solicitada. Isso não significa que todos esses agrupamentos sejam igualmente úteis.

In [ ]:
comparacao_k = []

for k in [2, 4, 7]:
    modelo = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=20
    )

    rotulos = modelo.fit_predict(X_padronizado)

    comparacao_k.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhouette": silhouette_score(X_padronizado, rotulos),
        "menor_cluster": pd.Series(rotulos).value_counts().min(),
        "maior_cluster": pd.Series(rotulos).value_counts().max()
    })

pd.DataFrame(comparacao_k).round(3)

### Pergunta 10

Compare os três valores de `k`.

Por que não seria adequado escolher simplesmente o modelo com menor inércia?

O agrupamento com maior silhouette é automaticamente o melhor para qualquer aplicação?

# 15. Exercícios

## Exercício 1 — efeito da ausência de padronização

Utilize `k=4` nos dados **sem padronização**.

Calcule o perfil médio dos clusters encontrados e compare com `perfil_clusters`.

Responda:

1. quais atributos parecem ter exercido maior influência?
2. os grupos continuam igualmente interpretáveis?
3. por que isso ocorreu?

In [ ]:
# Complete aqui.

## Exercício 2 — escolha de `k`

Teste `k` de 2 até 10.

Para cada valor, registre:

- inércia;
- silhouette média;
- tamanho do menor cluster.

Depois construa uma tabela ordenada pela silhouette.

In [ ]:
# Complete aqui.

## Exercício 3 — caracterização do cluster

Escolha um cluster do modelo com `k=4`.

Calcule para ele:

- mediana dos seis atributos;
- tamanho do grupo;
- silhouette média.

Compare média e mediana.

Existe algum atributo em que elas diferem de forma relevante?

In [ ]:
# Complete aqui.

## Exercício 4 — novos clientes

Considere:

```python
novo_cliente_1 = [5000, 1000, 8, 40, 3000, 5]
novo_cliente_2 = [15500, 6800, 30, 5, 39000, 26]
```

Utilize o `scaler` já ajustado e o modelo `kmeans` para descobrir em qual cluster cada novo cliente seria colocado.

Depois compare seus atributos com o perfil médio desses clusters.

In [ ]:
# Complete aqui.

# 16. Desafio final — relatório de segmentação

Você deve apresentar uma síntese para a área de relacionamento da VGAbank.

**Número de clusters adotado:** ______________________________

**Evidências utilizadas para escolher esse número:**  
_____________________________________________________________

**Cluster mais numeroso:** ___________________________________

**Cluster com maior renda média:** ____________________________

**Cluster com maior uso médio do aplicativo:** ________________

**Cluster com maior recência média:** _________________________

Agora descreva cada segmento em uma frase:

**Cluster 0:**  
_____________________________________________________________

**Cluster 1:**  
_____________________________________________________________

**Cluster 2:**  
_____________________________________________________________

**Cluster 3:**  
_____________________________________________________________

**Que ação poderia ser investigada para um dos grupos?**  
_____________________________________________________________

**Por que o agrupamento, sozinho, não prova que essa ação produzirá resultado?**  
_____________________________________________________________

# 17. Reflexão final

Responda brevemente:

**1. Qual é a principal diferença entre classificação e clustering?**

**2. Qual é o papel dos centroides no K-Means?**

**3. Por que a padronização pode alterar completamente o agrupamento?**

**4. Por que a inércia tende a diminuir quando aumentamos `k`?**

**5. O que uma silhouette próxima de 1 sugere?**

**6. O valor de `k` deve ser escolhido apenas pela maior silhouette?**

**7. Por que os números dos clusters não devem ser interpretados como categorias ordenadas?**

**8. Qual é a importância de construir um perfil dos clusters após executar o algoritmo?**

# 18. Conclusão

Nesta aula, percorremos o processo:

**dados sem rótulo → escala → K-Means → escolha de `k` → avaliação interna → visualização → perfil dos clusters → interpretação**

A principal mudança em relação às aulas anteriores é que não existe uma classe correta disponível para comparar diretamente com a saída do modelo.

Por isso, clustering exige combinar:

- critérios matemáticos;
- visualização;
- coesão e separação dos grupos;
- conhecimento do domínio;
- utilidade prática da segmentação.

> **Encontrar clusters é uma etapa matemática. Transformá-los em conhecimento é uma etapa analítica.**